In [2]:
import roboticstoolbox as rtb
import numpy as np

puma = rtb.models.DH.Puma560()
q_test = [0, -np.pi/4, np.pi/4, 0, np.pi/4, 0]

# 计算基坐标系下的 6x6 几何雅可比矩阵
J = puma.jacob0(q_test)
print("雅可比矩阵 J(q):\n", np.round(J, 4))

雅可比矩阵 J(q):
 [[ 0.1501 -0.1265 -0.4318  0.      0.      0.    ]
 [ 0.3256  0.      0.      0.      0.      0.    ]
 [ 0.      0.3256  0.0203  0.      0.      0.    ]
 [-0.      0.      0.     -0.      0.     -0.7071]
 [ 0.     -1.     -1.      0.     -1.     -0.    ]
 [ 1.      0.      0.      1.      0.      0.7071]]


In [3]:
# 设置奇异位形（θ5 = 0，手腕奇异）
q_singular = [0, 0, 0, 0, 0, 0]
J_sing = puma.jacob0(q_singular)

U, S, Vt = np.linalg.svd(J_sing)
manipulability = np.sqrt(np.linalg.det(J_sing @ J_sing.T))

print("奇异位形下的奇异值：", np.round(S, 4))
print(f"可操作度（Yoshikawa指标）：{manipulability:.4f}")

奇异位形下的奇异值： [1.8307 1.7526 0.4443 0.362  0.231  0.    ]
可操作度（Yoshikawa指标）：0.0000


In [4]:
dx = np.array([0.1, 0, 0, 0, 0, 0])  # 沿 X 轴移动 0.1 m/s

# 1. 普通伪逆法
dq_pinv = np.linalg.pinv(J_sing) @ dx

# 2. 阻尼最小二乘法 (DLS)
lambda_val = 0.1
dls_matrix = J_sing.T @ np.linalg.inv(J_sing @ J_sing.T + (lambda_val**2) * np.eye(6))
dq_dls = dls_matrix @ dx

print("普通伪逆求得的关节速度：\n", np.round(dq_pinv, 2))
print("\nDLS求得的关节速度：\n", np.round(dq_dls, 2))
print("\n结论：普通伪逆在奇异点会产生极大速度，DLS方法使速度保持在安全范围内。")

普通伪逆求得的关节速度：
 [-0.    0.01 -0.24  0.    0.23  0.  ]

DLS求得的关节速度：
 [ 0.01 -0.   -0.21 -0.    0.2  -0.  ]

结论：普通伪逆在奇异点会产生极大速度，DLS方法使速度保持在安全范围内。
